# Cattle Re-ID — Etapa 3: evaluación final (mcs=4, batería completa)

Re-evaluación determinista de todos los checkpoints (no reentrena nada). Reusa `scripts/reid_eval.py`.
**min_cluster_size = 4** (el mínimo real de fotos por vaca en Zenodo). Selección de eps **label-free**
(silhouette, grid ≤ 0.05, sin floor). Produce:
- **Fig loss** de DINOv2-large (el encoder que usamos).
- **T1** comparativa por loss (ResNet-50, aug strong): CE / ArcFace / SupCon / Triplet.
- **T2** comparativa por modelo (familia SupCon + variaciones de aug + baselines no entrenados).
- Todas las métricas de `reid_eval.py` (ARI, AMI, NMI, homog, compl, V, BCubed P/R/F1, #clust, ratio, noise).

Todo se guarda en Drive (`outputs/results/eval_final/`).

## 1. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/cattle_reid')
assert DRIVE_ROOT.is_dir(), f'No existe {DRIVE_ROOT}.'
print('contenido:', [p.name for p in DRIVE_ROOT.iterdir()])

## 2. Cachear modelos en Drive (antes de importar torch)

In [ ]:
import os
CACHE = DRIVE_ROOT / 'model_cache'
(CACHE/'torch').mkdir(parents=True, exist_ok=True)
(CACHE/'hf').mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(CACHE/'torch')
os.environ['HF_HOME']    = str(CACHE/'hf')
print('cache ->', CACHE)

## 3. GPU

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'Sin GPU: Entorno -> Cambiar tipo -> GPU.'
print('GPU:', torch.cuda.get_device_name(0))

## 4. Código + dependencias

In [ ]:
import shutil
os.chdir('/content')
REPO_URL = 'https://github.com/benjaminvitale/tp-final-vision2-Pujol-Vitale.git'
REPO_DIR = '/content/tp-final-vision2-Pujol-Vitale'
if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone -b main {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1
!pip -q install 'transformers==4.40.2' seaborn
import sys; sys.path.insert(0, 'scripts')   # para importar reid_eval

## 5. Persistir outputs en Drive (symlinks)

In [ ]:
for sub in ('checkpoints', 'logs', 'results'):
    ds = DRIVE_ROOT / 'outputs' / sub; ds.mkdir(parents=True, exist_ok=True)
    ls = Path(REPO_DIR) / 'outputs' / sub
    if ls.exists() and not ls.is_symlink(): shutil.rmtree(ls)
    if not ls.exists(): os.symlink(ds, ls, target_is_directory=True)
EVALDIR = Path('outputs/results/eval_final'); EVALDIR.mkdir(parents=True, exist_ok=True)
print('salidas ->', DRIVE_ROOT / 'outputs' / 'results' / 'eval_final')

## 6. Target de evaluación (Zenodo, pHash dedup) + índice reproducible

In [ ]:
import numpy as np, csv
IMG_EXTS = {'.jpg','.jpeg','.png','.bmp','.webp'}
MUZZLE_ZIP = DRIVE_ROOT / 'BeefCattle_Muzzle_Individualized.zip'
MUZZLE_LOCAL = Path('/content/muzzle')
if not MUZZLE_LOCAL.exists():
    MUZZLE_LOCAL.mkdir(parents=True); !unzip -q "{MUZZLE_ZIP}" -d "{MUZZLE_LOCAL}"
def find_root(base):
    for p in [base, *[d for d in base.rglob('*') if d.is_dir()]]:
        subs = [d for d in p.iterdir() if d.is_dir()]
        if len(subs) >= 100 and any(d.name.lower().startswith('cattle') for d in subs): return p
    raise RuntimeError('No encuentro cattle_*')
TARGET_DIR = find_root(MUZZLE_LOCAL)

from src.reid.reid_dataset import entries_from_folders
from src.reid.phash import dedup_entries
entries, _ = entries_from_folders(TARGET_DIR)
entries, dinfo = dedup_entries(entries, TARGET_DIR, threshold=6)
lab = np.array([e['label'] for e in entries]); n_true = len(np.unique(lab))
print(f'eval: {len(entries)} imgs | {n_true} ids | dedup={dinfo}')
# indice reproducible
idx_csv = EVALDIR / 'zenodo_1554_index.csv'
with open(idx_csv, 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['path', 'label'])
    for e in entries: w.writerow([str(e['path']), e['label']])
print('indice guardado ->', idx_csv)

## 7. Config de evaluación

In [ ]:
MCS = 4                                    # min_cluster_size (min real de fotos/vaca en Zenodo)
EPS_GRID = list(np.round(np.arange(0.0, 0.051, 0.005), 3))   # grid <= 0.05, sin floor (no colapsa)
SEED = 0
print('MCS =', MCS, '| EPS_GRID =', EPS_GRID)

## 8. Sanity gate (a mcs=2, para confirmar que el pipeline reproduce lo conocido)

A mcs=2 deben salir ~0.461 (imagenet) y ~0.716 (dinov2-large). Si no, el pipeline no replica.

In [ ]:
from sklearn.cluster import HDBSCAN
from src.reid.encoders import resnet50_checkpoint, dinov2_checkpoint, build_encoder
from src.reid.cluster import clustering_metrics

def embed(enc):
    e, l = enc.embed(entries, data_dir=TARGET_DIR, batch_size=64, num_workers=2)
    return e, l

for name, enc, exp in [('imagenet', build_encoder('imagenet'), 0.461),
                       ('dinov2L', dinov2_checkpoint('outputs/checkpoints/cmpd300_supcon_dinov2L_strong.pt'), 0.716)]:
    e, l = embed(enc)
    ari = clustering_metrics(l, HDBSCAN(min_cluster_size=2, metric='cosine').fit_predict(e))['ARI']
    print(f'  {name:10} ARI@eps0(mcs=2) = {ari:.3f}  (esperado ~{exp})  {"OK" if abs(ari-exp)<0.02 else "REVISAR"}')

## 9. Evaluar todos los modelos a mcs=4 (batería completa)

In [ ]:
import json
from reid_eval import full_metrics, eps_sweep
from src.reid.eval_reid import rank_metrics
from src.reid.cluster import kmeans_reference
import random

def single_shot_idx(labels, seed=0):
    rng = random.Random(seed); by = {}
    for i, l in enumerate(labels.tolist()): by.setdefault(l, []).append(i)
    gal, prb = [], []
    for _l, idxs in sorted(by.items()):
        if len(idxs) < 2: continue
        idxs = idxs[:]; rng.shuffle(idxs); gal.append(idxs[0]); prb += idxs[1:]
    return gal, prb
GAL, PRB = single_shot_idx(lab, SEED)

def loader(ckpt, base):
    if ckpt is None:
        return build_encoder('dinov2' if base == 'dinov2' else 'imagenet')
    return dinov2_checkpoint(ckpt) if base == 'dinov2' else resnet50_checkpoint(ckpt)

# (display, ckpt|None, base, aug, grupo)  grupo: 'loss' | 'model' | 'both' | 'baseline'
ALL_MODELS = [
    ('ImageNet (frozen)',            None,                                            'resnet', '-',      'baseline'),
    ('DINOv2-base (frozen)',         None,                                            'dinov2', '-',      'baseline'),
    ('CE',                           'outputs/checkpoints/cmpd300_ce.pt',             'resnet', 'strong', 'loss'),
    ('ArcFace',                      'outputs/checkpoints/cmpd300_arcface.pt',        'resnet', 'strong', 'loss'),
    ('Triplet',                      'outputs/checkpoints/cmpd300_triplet.pt',        'resnet', 'strong', 'loss'),
    ('SupCon (ResNet, strong)',      'outputs/checkpoints/cmpd300_supcon.pt',         'resnet', 'strong', 'both'),
    ('SupCon (ResNet, heavy)',       'outputs/checkpoints/cmpd300_supcon_heavyaug.pt','resnet', 'heavy',  'model'),
    ('SupCon (DINOv2-base, strong)', 'outputs/checkpoints/cmpd300_supcon_dinov2_strong.pt',   'dinov2', 'strong', 'model'),
    ('SupCon (DINOv2-base, heavy)',  'outputs/checkpoints/cmpd300_supcon_dinov2_heavyaug.pt', 'dinov2', 'heavy',  'model'),
    ('SupCon (DINOv2-large, strong)','outputs/checkpoints/cmpd300_supcon_dinov2L_strong.pt',  'dinov2', 'strong', 'model'),
]

RESULTS = {}
for disp, ckpt, base, aug, grupo in ALL_MODELS:
    if ckpt is not None and not Path(ckpt).is_file():
        print(f'  FALTA {disp} ({ckpt}) -> salteo'); continue
    emb, lb = embed(loader(ckpt, base))
    rows, eps_star = eps_sweep(emb, y_true=lb, eps_grid=EPS_GRID, min_cluster_size=MCS)  # label-free
    y0 = HDBSCAN(min_cluster_size=MCS, metric='cosine').fit_predict(emb)
    ys = HDBSCAN(min_cluster_size=MCS, metric='cosine', cluster_selection_epsilon=float(eps_star)).fit_predict(emb)
    m0, ms = full_metrics(lb, y0), full_metrics(lb, ys)
    rk = rank_metrics(emb[PRB], lb[PRB], emb[GAL], lb[GAL])
    kmeans_ari = full_metrics(lb, kmeans_reference(emb, k=n_true, seed=SEED))['ARI']
    RESULTS[disp] = {'base': base, 'aug': aug, 'grupo': grupo, 'eps_star': float(eps_star),
                     'ari_eps0': m0['ARI'], 'eps0': m0, 'epsstar': ms,
                     'rank1': round(rk['rank1'], 4), 'kmeans_ari': round(kmeans_ari, 4),
                     'eps_rows': rows}
    print(f"  {disp:32} ARI@eps0={m0['ARI']:.3f}  eps*={eps_star:.3f}  ARI@eps*={ms['ARI']:.3f}  noise={ms['noise_frac']:.2f}")

(EVALDIR / 'metrics_all.json').write_text(json.dumps(RESULTS, indent=2, default=float))
print('\nmetricas guardadas ->', EVALDIR / 'metrics_all.json')

## 10. Tablas (todas las métricas de reid_eval, en eps\*) → Markdown a Drive

Cada tabla reporta la batería completa en la config **eps\*** (label-free, reportable), más
`ARI@eps0` y `eps*` de referencia, y Rank-1 / k-means (de `src/reid/`).

In [ ]:
FULL = [('ARI','{:.3f}'), ('AMI','{:.3f}'), ('NMI','{:.3f}'), ('homogeneity','{:.3f}'),
        ('completeness','{:.3f}'), ('v_measure','{:.3f}'), ('bcubed_P','{:.3f}'),
        ('bcubed_R','{:.3f}'), ('bcubed_F1','{:.3f}'), ('n_clusters','{:d}'),
        ('cluster_ratio','{:.2f}'), ('noise_frac','{:.2f}')]
HEAD = ['ARI(eps0)', 'eps*'] + [c for c, _ in FULL] + ['Rank-1', 'kmeans']

def md_table(names):
    cols = '| modelo | ' + ' | '.join(HEAD) + ' |'
    sep = '|' + '---|' * (len(HEAD) + 1)
    lines = [cols, sep]
    for nm in names:
        r = RESULTS[nm]; ms = r['epsstar']
        cells = [f"{r['ari_eps0']:.3f}", f"{r['eps_star']:.3f}"]
        cells += [fmt.format(ms[c]) for c, fmt in FULL]
        cells += [f"{r['rank1']:.3f}", f"{r['kmeans_ari']:.3f}"]
        lines.append('| ' + nm + ' | ' + ' | '.join(cells) + ' |')
    return '\n'.join(lines)

T1_names = [n for n in RESULTS if RESULTS[n]['grupo'] in ('loss', 'both')]
T2_names = [n for n in RESULTS if RESULTS[n]['grupo'] in ('baseline', 'model', 'both')]

T1 = '## T1 — Comparación por loss (ResNet-50, aug strong, mcs=4)\n\n' + md_table(T1_names)
T2 = '## T2 — Comparación por modelo (familia SupCon + aug + baselines, mcs=4)\n\n' + md_table(T2_names)
(EVALDIR / 'T1_losses.md').write_text(T1)
(EVALDIR / 'T2_models.md').write_text(T2)
print(T1, '\n\n', T2)
print('\ntablas ->', EVALDIR)

## 11. Figura — curva de loss de DINOv2-large (el encoder que usamos)

In [ ]:
import re, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
logf = Path('outputs/logs/train_supcon_dinov2L_strong.log')
ep, loss = [], []
if logf.exists():
    for line in logf.read_text().splitlines():
        m = re.search(r'ep (\d+)/\d+ \| loss ([\d.]+)', line)
        if m: ep.append(int(m.group(1))); loss.append(float(m.group(2)))
if ep:
    plt.figure(figsize=(8, 4.5))
    plt.plot(ep, loss, color='#2b6cb0', linewidth=2, marker='o', markersize=3)
    plt.xlabel('Época'); plt.ylabel('SupCon loss (train)')
    plt.title('DINOv2-large + SupCon — curva de entrenamiento', fontweight='bold')
    plt.grid(alpha=0.3); plt.tight_layout()
    out = EVALDIR / 'fig_loss_dinov2L.png'; plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
    print(f'curva: {len(ep)} epochs, loss {loss[0]:.3f} -> {loss[-1]:.3f}  ->', out)
    plateau = abs(loss[-1] - loss[-5]) < 0.01 if len(loss) >= 5 else None
    print('plateó al final?' , plateau)
else:
    print('no encontre el log train_supcon_dinov2L_strong.log en Drive')

## 12. Figuras de las tablas (escalera por modelo + barrido de eps del ganador)

In [ ]:
from reid_eval import plot_encoder_staircase, plot_eps_sweep
# escalera por modelo (ARI@eps*)
res_star = {n: RESULTS[n]['epsstar'] for n in T2_names}
plot_encoder_staircase(res_star, metric='ARI', out=str(EVALDIR / 'fig_staircase_models.png'))
# barrido de eps del ganador
plot_eps_sweep(RESULTS['SupCon (DINOv2-large, strong)']['eps_rows'],
               eps_star=RESULTS['SupCon (DINOv2-large, strong)']['eps_star'],
               out=str(EVALDIR / 'fig_eps_sweep_dinov2L.png'))
print('figuras guardadas en', EVALDIR)

## 13. Verificación — qué quedó en Drive

In [ ]:
saved = sorted(EVALDIR.glob('*'))
for p in saved:
    print(f'  {p.name:32} {p.stat().st_size/1024:.0f} KB')
print('\nTODO en', EVALDIR)